# MOSS-Image SFT LoRA Trainer for Kaggle (2× T4)

This notebook runs **MOSS-Image SFT** with AI Toolkit's separate **Moss-Image SFT-Kaggle** architecture entry. It launches two data-parallel workers: each T4 holds a complete model copy, trains on a different batch, and synchronizes LoRA gradients.

**Before running:**
1. In Kaggle Settings, select **GPU T4 x2** and enable **Internet**.
2. Attach a Kaggle Dataset containing images and optional matching `.txt` captions.
3. In the settings cell, set `DATASET_DIR`, `LORA_NAME`, and any training options.

Each optimizer step processes one batch on each GPU (global batch size 2 with the defaults). This does not split model weights across GPUs; the model must fit on each T4 independently.

**Workflow:** Setup > Settings & Dataset > Train (or Resume) > Package > User Guide


In [ ]:
import os, sys, subprocess, shutil, time, json, re, glob, psutil

# Keep both Kaggle GPUs visible. Do not set CUDA_VISIBLE_DEVICES here.
os.environ["MODELS_PATH"] = "/tmp/models"
os.environ["HF_HOME"] = "/tmp/hf"
os.environ["HF_HUB_CACHE"] = "/tmp/hf/hub"
os.environ["HF_XET_CACHE"] = "/tmp/hf/xet"
os.environ["XDG_CACHE_HOME"] = "/tmp/cache"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTHONUNBUFFERED"] = "1"

for path in ("/tmp/models", "/tmp/hf/hub", "/tmp/hf/xet", "/tmp/cache"):
    os.makedirs(path, exist_ok=True)

def run_cmd(args, cwd=None, check=True, quiet=False):
    result = subprocess.run(args, cwd=cwd, text=True, capture_output=True)
    if not quiet and result.stdout:
        print(result.stdout[-4000:])
    if result.returncode and not quiet:
        print(result.stderr[-4000:])
    if check and result.returncode:
        details = (result.stderr or result.stdout)[-12000:]
        raise RuntimeError(
            f"Command failed ({result.returncode}): {' '.join(args)}\n\n{details}"
        )
    return result

print("=== Kaggle GPU Check ===")
gpu_info = run_cmd(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], quiet=True)
gpu_names = [line.strip() for line in gpu_info.stdout.splitlines() if line.strip()]
print("\n".join(gpu_names))
if len(gpu_names) < 2:
    raise RuntimeError("Two visible GPUs are required. Set Kaggle Accelerator to GPU T4 x2 and rerun this cell.")
print(f"System RAM: {psutil.virtual_memory().total/1e9:.1f} GB")
print(f"Working disk free: {shutil.disk_usage('/kaggle/working').free/1e9:.1f} GB")
print(f"Temporary disk free: {shutil.disk_usage('/tmp').free/1e9:.1f} GB")

TOOLKIT = "/kaggle/working/ai-toolkit-dual-gpu"
REPO_URL = "https://github.com/xelionaj/ai-toolkit-dual-gpu.git"
if os.path.isdir(os.path.join(TOOLKIT, ".git")):
    run_cmd(["git", "-C", TOOLKIT, "pull", "--ff-only", "origin", "main"])
else:
    run_cmd(["git", "clone", "--branch", "main", REPO_URL, TOOLKIT])
run_cmd(["git", "submodule", "update", "--init", "--recursive"], cwd=TOOLKIT)

# Keep Kaggle's CUDA-enabled PyTorch; flatten included requirement files and
# choose the matching TorchAO release for that installed PyTorch version.
import torch
print("Installing AI Toolkit dependencies (preserving Kaggle's installed PyTorch)...")
print(f"Kaggle PyTorch: {torch.__version__}")
run_cmd([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools", "wheel"])
requirements = os.path.join(TOOLKIT, "requirements.txt")
clean_requirements = "/tmp/requirements_ai_toolkit_kaggle.txt"
filtered_requirements = []

def add_requirements(requirements_path, active_paths=None):
    active_paths = active_paths or set()
    requirements_path = os.path.abspath(requirements_path)
    if requirements_path in active_paths:
        raise RuntimeError(f"Circular requirement include: {requirements_path}")
    active_paths.add(requirements_path)
    with open(requirements_path, "r", encoding="utf-8") as src:
        for line in src:
            stripped = line.strip()
            if not stripped or stripped.startswith("#"):
                continue
            if stripped.startswith(("-r ", "--requirement ")):
                include_path = stripped.split(maxsplit=1)[1]
                if not os.path.isabs(include_path):
                    include_path = os.path.join(os.path.dirname(requirements_path), include_path)
                add_requirements(include_path, active_paths)
                continue
            package = stripped.lower()
            package_name = re.split(r"[<>=!~;\[]", package, maxsplit=1)[0].strip()
            if package_name in ("scipy", "torch", "torchvision", "torchaudio", "torchao"):
                continue
            filtered_requirements.append(stripped)
    active_paths.remove(requirements_path)

add_requirements(requirements)
torch_version = tuple(int(part) for part in torch.__version__.split("+")[0].split(".")[:2])
if torch_version >= (2, 11):
    torchao_version = "0.17.0"
elif torch_version >= (2, 10):
    torchao_version = "0.16.0"
elif torch_version >= (2, 9):
    torchao_version = "0.14.1"
elif torch_version >= (2, 8):
    torchao_version = "0.13.0"
elif torch_version >= (2, 7):
    torchao_version = "0.12.0"
else:
    raise RuntimeError(f"Unsupported Kaggle PyTorch version for this notebook: {torch.__version__}")
filtered_requirements.append(f"torchao=={torchao_version}")
with open(clean_requirements, "w", encoding="utf-8") as dst:
    dst.write("\n".join(filtered_requirements) + "\n")
print(f"Using torchao=={torchao_version} for PyTorch {torch.__version__}")
install = run_cmd([sys.executable, "-m", "pip", "install", "-q", "--only-binary", ":all:", "-r", clean_requirements], check=False, quiet=True)
if install.returncode:
    run_cmd([sys.executable, "-m", "pip", "install", "-q", "-r", clean_requirements])

print(f"PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}")
print(f"Visible CUDA devices: {torch.cuda.device_count()}")
for index in range(torch.cuda.device_count()):
    print(f"  GPU {index}: {torch.cuda.get_device_name(index)}")
if torch.cuda.device_count() < 2:
    raise RuntimeError("PyTorch sees fewer than two GPUs. Check Kaggle's GPU T4 x2 setting.")
if shutil.which("accelerate") is None:
    raise RuntimeError("The accelerate command was not installed with AI Toolkit requirements.")
print("Setup completed. Both GPUs are available to the distributed launcher.")


## Settings & Dataset Preparation
Review each parameter below. Sample values and rules are documented directly in the comments.

In [ ]:
# =============================================================================
# 1. DATASET & IDENTIFIERS
# =============================================================================

# Path to folder containing image files (.jpg, .png, etc.) and optional .txt captions.
# NOTE: In Kaggle, datasets are mounted as '/kaggle/input/<dataset-slug>'
# Samples: "/kaggle/input/mydata2", "/kaggle/input/cyberpunk-art-style"
DATASET_DIR = "/kaggle/input/YOUR-DATASET-NAME"

# Filename identifier for generated checkpoints and output folder.
# Rule: Lowercase, numbers, underscores, or hyphens (avoid spaces).
# Samples: "my_moss_lora", "vintage_anime_v1", "character_john_doe"
LORA_NAME = "my_moss_lora"

# Unique keyword used in generation prompts to activate this LoRA concept.
# Samples:
#   - ""          -> (Disabled) Best for general styles or fine-tunes
#   - "ohwx"      -> Rare pseudo-word to isolate a specific person/character
#   - "cptstyle"  -> Custom token for a specific art/illustration style
TRIGGER_WORD = ""

# =============================================================================
# 2. TRAINING HYPERPARAMETERS
# =============================================================================

# LoRA Rank (r) & Alpha: Capacity of the adapter.
# Samples:
#   - 16 : Lightweight, faster training, lowest VRAM
#   - 32 : Recommended default; balances expressiveness with generalization
#   - 64 : High detail capture, useful for complex characters
RANK = 32

# Learning Rate: Rate of weight updates with 8-bit AdamW.
# Samples:
#   - 5e-5  (0.00005) : Conservative/safe; minimizes risk of burning details
#   - 1e-4  (0.00010) : Recommended default; reliable convergence for MOSS-Image
#   - 1.5e-4(0.00015) : Slightly faster learning; best if batch_size >= 2
LR = 1e-4

# Total Cumulative Training Steps:
# NOTE: In AI Toolkit, this is the CUMULATIVE target step count.
# Guidance:
#   - 3 to 10 images   : 200 - 350 steps   (~40-60 epochs; avoids burning)
#   - 15 to 30 images  : 600 - 1000 steps  (~30-40 epochs)
#   - 40 to 100 images : 1200 - 2000 steps (~20-30 epochs)
TRAIN_STEPS = 300

# Checkpoint Saving Frequency (steps):
# Samples:
#   - 50  : Recommended for short runs (200-400 steps)
#   - 100 : Recommended for medium runs (500-1000 steps)
#   - 250 : Recommended for long runs (1000+ steps)
SAVE_EVERY = 50

# Target Resolution for Aspect Ratio Bucketing:
# Non-square images are automatically mapped to equivalent pixel areas without stretching.
# Samples:
#   - [1024]       : Full-resolution MOSS training (higher VRAM and slower than 512px)
#   - [512]        : Lower-memory fallback
RESOLUTIONS = [1024]

# =============================================================================
# 3. RESUMPTION & BASE MODEL CONFIGURATION
# =============================================================================

# Resumption Path:
# Samples:
#   - "" : Auto-resumes from /kaggle/working/output/ if checkpoints exist, else fresh start
#   - "/kaggle/input/prev-run/output/my_moss_lora" : Directory from previous run
#   - "/kaggle/input/prev-run/my_moss_lora_000000150.safetensors" : Specific checkpoint
RESUME_FROM = ""

MODEL_ID   = "OpenMOSS-Team/MOSS-Image-SFT"
MODEL_ARCH = "moss_image_kaggle"

# =============================================================================
# 4. DATASET VERIFICATION & PRESERVATION
# =============================================================================
IMG_EXT = (".jpg", ".jpeg", ".png", ".webp")
DATASET_COPY = "/tmp/dataset"

assert os.path.exists(DATASET_DIR), f"ERROR: Dataset path not found: '{DATASET_DIR}'. Check DATASET_DIR."

# Locate images (handles nested folders automatically)
src_images = []
for root, _, files in os.walk(DATASET_DIR):
    for f in files:
        if f.lower().endswith(IMG_EXT):
            src_images.append(os.path.join(root, f))

assert src_images, f"ERROR: No valid images found inside '{DATASET_DIR}'."

# Copy into /tmp while preserving nested folders, avoiding filename collisions.
os.makedirs(DATASET_COPY, exist_ok=True)
for img_path in src_images:
    rel_path = os.path.relpath(img_path, DATASET_DIR)
    dest_img = os.path.join(DATASET_COPY, rel_path)
    os.makedirs(os.path.dirname(dest_img), exist_ok=True)
    shutil.copy2(img_path, dest_img)

    # Copy matching caption if present
    src_txt = os.path.splitext(img_path)[0] + ".txt"
    dest_txt = os.path.splitext(dest_img)[0] + ".txt"
    if os.path.exists(src_txt):
        shutil.copy2(src_txt, dest_txt)

# Auto-heal missing captions (prevents unhandled FileNotFoundError in AI Toolkit)
synced_imgs = []
for root, _, files in os.walk(DATASET_COPY):
    synced_imgs.extend(os.path.join(root, name) for name in files if name.lower().endswith(IMG_EXT))
created_captions = 0
for img_path in synced_imgs:
    txt_file = os.path.splitext(img_path)[0] + ".txt"
    if not os.path.exists(txt_file):
        with open(txt_file, "w", encoding="utf-8") as f:
            f.write(TRIGGER_WORD if TRIGGER_WORD else "")
        created_captions += 1

print(f"Dataset verified: {len(synced_imgs)} images ready in {DATASET_COPY}.")
if created_captions > 0:
    print(f"Notice: Auto-generated {created_captions} missing .txt caption files (populated with trigger word).\n")

images_per_optimizer_step = 2  # batch_size=1 on each of the two GPUs
epochs = TRAIN_STEPS * images_per_optimizer_step / len(synced_imgs)
print(f"Target: {TRAIN_STEPS} cumulative optimizer steps")
print(f"Approximate dataset passes: {epochs:.1f} over {len(synced_imgs)} images (2 samples per optimizer step)")

## Train / Resume LoRA
Generates the configuration and starts training. Features graceful process cleanup on interrupt, unbuffered logging, and cumulative step checking.

In [ ]:
import yaml
import signal

OUT_DIR = "/kaggle/working/output"
lora_dir = os.path.join(OUT_DIR, LORA_NAME)
os.makedirs(lora_dir, exist_ok=True)
os.makedirs("/kaggle/working/configs", exist_ok=True)
os.makedirs("/kaggle/working/logs", exist_ok=True)

# Import previous run checkpoints and optimizer states if RESUME_FROM is specified
if RESUME_FROM and os.path.exists(RESUME_FROM):
    print(f"Importing resumption files from: {RESUME_FROM}")
    if os.path.isdir(RESUME_FROM):
        for item in glob.glob(os.path.join(RESUME_FROM, "*")):
            dest = os.path.join(lora_dir, os.path.basename(item))
            if not os.path.exists(dest):
                shutil.copy2(item, dest)
    elif os.path.isfile(RESUME_FROM):
        dest = os.path.join(lora_dir, os.path.basename(RESUME_FROM))
        if not os.path.exists(dest):
            shutil.copy2(RESUME_FROM, dest)

# Check existing checkpoints & validate TRAIN_STEPS
existing_ckpts = sorted(glob.glob(os.path.join(lora_dir, "*.safetensors")), key=os.path.getmtime)
if existing_ckpts:
    latest_ckpt = os.path.basename(existing_ckpts[-1])
    step_match = re.search(r"_(\d+)\.safetensors", latest_ckpt)
    if step_match:
        resumed_step = int(step_match.group(1))
        print(f"\n[RESUME MODE] Found checkpoint '{latest_ckpt}' (Step {resumed_step}).")
        if resumed_step >= TRAIN_STEPS:
            raise ValueError(
                f"TRAIN_STEPS is set to {TRAIN_STEPS}, but the existing checkpoint is already at step {resumed_step}. "
                f"In AI Toolkit, TRAIN_STEPS is cumulative. Increase TRAIN_STEPS to at least {resumed_step + 50} in Settings!"
            )
    else:
        print(f"\n[RESUME MODE] Found checkpoint '{latest_ckpt}'. Resuming from latest state.")
else:
    print("\n[FRESH START] No previous checkpoints found. Training starting from step 0.")

def gpu_used_gb():
    out = run_cmd(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"], quiet=True).stdout.strip()
    values = []
    for line in out.splitlines():
        try:
            values.append(f"{float(line.strip())/1024:.1f} GB")
        except ValueError:
            values.append("n/a")
    return " / ".join(values)

config = {
    "job": "extension",
    "config": {
        "name": LORA_NAME,
        "process": [{
            "type": "diffusion_trainer",
            "training_folder": OUT_DIR,
            "device": "cuda",
            "network": {
                "type": "lora",
                "linear": RANK,
                "linear_alpha": RANK
            },
            "save": {
                "dtype": "float16",
                "save_every": SAVE_EVERY,
                "max_step_saves_to_keep": 4
            },
            "datasets": [{
                "folder_path": DATASET_COPY,
                "caption_ext": "txt",
                "caption_dropout_rate": 0.0,
                "resolution": RESOLUTIONS,
                "cache_latents_to_disk": True,
                "cache_text_embeddings": True,
            }],
            "train": {
                "batch_size": 1,
                "steps": TRAIN_STEPS,
                "gradient_accumulation": 1,
                "train_unet": True,
                "train_refiner": False,
                "train_text_encoder": False,
                "gradient_checkpointing": True,
                "noise_scheduler": "flowmatch",
                "timestep_type": "shift",
                "optimizer": "adamw8bit",
                "lr": LR,
                "dtype": "fp16",
                "disable_sampling": True,
                "unload_text_encoder": True,
            },
            "model": {
                "name_or_path": MODEL_ID,
                "arch": MODEL_ARCH,
                "quantize": True,
                "qtype": "convrot8",
                "quantize_te": True,
                "qtype_te": "convrot8",
                "low_vram": True,
            },
        }]
    },
    "meta": {"name": "[name]", "version": "1.0"}
}

if TRIGGER_WORD:
    config["config"]["process"][0]["trigger_word"] = TRIGGER_WORD

cfg_path = f"/kaggle/working/configs/{LORA_NAME}.yaml"
yaml.safe_dump(config, open(cfg_path, "w"), sort_keys=False)
log_path = f"/kaggle/working/logs/{LORA_NAME}_train.log"

# Ensure the subprocess receives MODELS_PATH and all temp storage overrides
sub_env = os.environ.copy()
sub_env["MODELS_PATH"] = "/tmp/models"
sub_env["HF_HOME"] = "/tmp/hf"
sub_env["HF_HUB_CACHE"] = "/tmp/hf/hub"
sub_env["HF_XET_CACHE"] = "/tmp/hf/xet"
sub_env["XDG_CACHE_HOME"] = "/tmp/cache"
sub_env["PYTHONUNBUFFERED"] = "1"

print(f"\nLaunching AI Toolkit...")
log_file = open(log_path, "w")
p = subprocess.Popen(["accelerate", "launch", "--multi_gpu", "--num_processes", "2", "run.py", cfg_path], cwd=TOOLKIT,
                     stdout=log_file, stderr=subprocess.STDOUT, env=sub_env, start_new_session=True)

t0 = time.time()
try:
    # Quick initial check to catch early launch crashes
    time.sleep(5)
    if p.poll() is not None:
        log_file.close()
        raise RuntimeError(f"Process exited immediately. Check error log: {log_path}")

    while p.poll() is None:
        time.sleep(60)
        with open(log_path, "r", errors="ignore") as f:
            lines = f.read().replace("\r", "\n").strip().splitlines()
        status_line = lines[-1][:140] if lines else "Initializing..."
        print(f"[{(time.time()-t0)/60:.0f} min] VRAM GPU0/GPU1: {gpu_used_gb()} | RAM: {psutil.virtual_memory().used/1e9:.1f} GB | {status_line}")
except KeyboardInterrupt:
    print("\nInterrupt signal received! Terminating background training process...")
    try:
        os.killpg(p.pid, signal.SIGINT)
        p.wait(timeout=30)
    except (ProcessLookupError, subprocess.TimeoutExpired):
        try:
            os.killpg(p.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
        p.wait()
    print("Process terminated cleanly. GPU VRAM released. Checkpoints preserved in:", lora_dir)
finally:
    if not log_file.closed:
        log_file.close()

if p.returncode == 0:
    print(f"\nTraining completed successfully in {(time.time()-t0)/60:.1f} minutes!")
elif p.returncode not in (0, -15, -9):
    print(f"\nProcess exited with error code {p.returncode}. See log at: {log_path}")

## Package LoRA (Downloadable ZIP)
Collects your `.safetensors` checkpoints and creates `/kaggle/working/lora_out.zip`. Uses `/tmp` as a staging directory to prevent filling Kaggle's 20 GB working disk quota.

In [ ]:
staging_dir = "/tmp/lora_staging"
if os.path.exists(staging_dir):
    shutil.rmtree(staging_dir)
os.makedirs(staging_dir, exist_ok=True)

files = sorted(glob.glob(f"{OUT_DIR}/{LORA_NAME}/*.safetensors"), key=os.path.getmtime)
assert files, f"No .safetensors checkpoints found in {OUT_DIR}/{LORA_NAME}. Verify training logs."

print("Packaging LoRA checkpoints:")
for f in files:
    shutil.copy2(f, staging_dir)
    print(f" - {os.path.basename(f)} ({os.path.getsize(f) / 1e6:.1f} MB)")

# Archive directly from /tmp to /kaggle/working/lora_out.zip
zip_path = shutil.make_archive("/kaggle/working/lora_out", "zip", staging_dir)
# Remove staging directory to leave 0 unnecessary duplication
shutil.rmtree(staging_dir)

print(f"\nDone! Download your weights from the right panel:\n-> {zip_path} ({os.path.getsize(zip_path) / 1e6:.1f} MB)")

---
# User Guide & Best Practices

### 1. Download your checkpoints
In Kaggle's **Output** panel, download `lora_out.zip`. It contains the saved `.safetensors` checkpoints from `/kaggle/working/output/<LORA_NAME>`.

### 2. Resume in a later Kaggle session
1. Save the Kaggle notebook version and preserve its output, or download `lora_out.zip`.
2. In the new session, attach the previous output as an input dataset.
3. Set `RESUME_FROM` to the prior run folder or checkpoint file, keep `LORA_NAME` and `RANK` the same, and increase `TRAIN_STEPS` to the desired cumulative total.
4. Run the setup, settings, and training cells. This notebook copies the resume files into the new output folder before launch.

### 3. Two-GPU behavior
Both T4s train in data-parallel mode. Each GPU holds a complete copy of MOSS-Image SFT, sees different training samples, and shares averaged LoRA gradients. With `batch_size=1`, each optimizer step uses two samples globally. The model must fit on each GPU by itself; this approach does not combine GPU memory to fit a larger single model.

The training cell reports VRAM for GPU 0 and GPU 1. If either GPU is missing or the run exits early, inspect `/kaggle/working/logs/<LORA_NAME>_train.log`.

### 4. Resolution and quality
The default is 1024px. If a T4 runs out of VRAM, change `RESOLUTIONS` to `[512]` and restart the run. Test intermediate checkpoints as well as the final one; more steps can overfit a small dataset.
